# 22 — Hybrid Search RAG (BM25 + Vector)

Combine two complementary retrieval strategies — keyword-based BM25 and semantic vector search — into a single `EnsembleRetriever` that covers each strategy's blind spots.

**What you'll learn**
- Why pure vector search fails on exact identifiers (model numbers, names, codes)
- Why pure BM25 fails on paraphrased or conceptual queries
- `BM25Retriever` from `langchain_community` — bag-of-words statistical ranking
- `EnsembleRetriever` — Reciprocal Rank Fusion merges two ranked document lists
- `weights=[0.5, 0.5]` — how to tune BM25 vs vector balance for your domain

**Companion examples:** 1-basic-local-rag (vector-only baseline), 26-rag-fusion (RRF applied to query variants)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma langchain-community chromadb rank_bm25 python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Compare BM25 vs vector on the same queries ────────────────────────────
# Before wiring the full pipeline, test each approach independently.
# BM25: keyword matching — great on exact tokens, blind to semantics
# Vector: embedding similarity — great on paraphrases, fuzzy on exact identifiers
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings

K = 3

DOCS = [
    "The Apex-X200 is a high-performance SSD with 7,000 MB/s sequential read speed and NVMe Gen4 interface.",
    "The Apex-X100 is an entry-level SSD with 3,500 MB/s sequential read speed and NVMe Gen3 interface.",
    "The Apex-M50 is a mechanical hard drive with 7,200 RPM and 256 MB cache, designed for archival storage.",
    "The Vertex-Pro GPU supports ray tracing, DLSS 3.0, and has 16 GB of GDDR6X memory.",
    "The Vertex-Core GPU is an entry model with 8 GB GDDR6 memory, targeting 1080p gaming.",
    "Our return policy allows returns within 30 days of purchase for all products in original packaging.",
    "Warranty coverage: Apex-X200 and Apex-X100 carry a 5-year limited warranty. Apex-M50 carries 3 years.",
    "The Apex-X200 is compatible with PCIe 4.0 and PCIe 5.0 motherboards via backward compatibility.",
    "Vertex-Pro requires a 750W PSU minimum. Vertex-Core requires 550W minimum.",
    "Technical support for Apex series products is available 24/7 via support.apextech.com.",
]

bm25 = BM25Retriever.from_texts(DOCS)
bm25.k = K

vectorstore = Chroma.from_texts(
    DOCS, OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="hybrid_demo"
)
vector = vectorstore.as_retriever(search_kwargs={"k": K})

exact_query = "What is the read speed of the Apex-X200?"  # BM25 wins: exact token
semantic_query = "Which product is best for budget gaming?"  # Vector wins: semantic intent

print("=== Exact query — BM25 advantage ===")
for d in bm25.invoke(exact_query):
    print(f"  BM25: {d.page_content[:75]}")

print("\n=== Semantic query — Vector advantage ===")
for d in vector.invoke(semantic_query):
    print(f"  VEC:  {d.page_content[:75]}")

In [ ]:
# ── 4. EnsembleRetriever — Reciprocal Rank Fusion ─────────────────────────────
# RRF score per document: sum(1 / (rank + k)) across both ranked lists.
# A doc ranked #1 in BOTH lists scores ~0.5; ranked #10 in both scores ~0.1.
# A doc that only appears in ONE list still gets a partial score.
#
# weights=[0.5, 0.5] — equal importance. Raise BM25 for keyword-heavy corpora.
from langchain.retrievers import EnsembleRetriever

ensemble = EnsembleRetriever(
    retrievers=[bm25, vector],
    weights=[0.5, 0.5],
)

print("=== Hybrid retrieval on both queries ===")
for q in [exact_query, semantic_query]:
    docs = ensemble.invoke(q)
    print(f"\nQ: {q}")
    for d in docs:
        print(f"  -> {d.page_content[:75]}")

In [ ]:
# ── 5. Full RAG graph: retrieve → generate ────────────────────────────────────
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class HybridRAGState(TypedDict):
    question: str
    documents: list
    answer: str


def retrieve(state: HybridRAGState) -> dict:
    docs = ensemble.invoke(state["question"])
    return {"documents": [d.page_content for d in docs]}


def generate(state: HybridRAGState) -> dict:
    context = "\n\n".join(state["documents"])
    response = llm.invoke(
        [
            SystemMessage(
                f"Answer using only the context below. If not in context, say so.\n\nContext:\n{context}"
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"answer": response.content}


graph = StateGraph(HybridRAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
app = graph.compile()

SAMPLE_QUESTIONS = [
    "What is the read speed of the Apex-X200?",
    "Which product is best for budget gaming?",
    "What warranty does the Apex-M50 come with?",
    "Can I return a product if I change my mind?",
]

for q in SAMPLE_QUESTIONS:
    result = app.invoke({"question": q, "documents": [], "answer": ""})
    print(f"Q: {q}")
    print(f"A: {result['answer']}\n")

## Exercises

**Exercise 1 — Tune weights:** Change `weights=[0.7, 0.3]` to favour BM25. Ask `"Which product is best for budget gaming?"`. Does retrieval quality drop for semantic queries?

**Exercise 2 — Keyword blind spot:** Remove `"Vertex-Core"` from DOCS and rebuild. Ask `"What is the entry-level GPU?"`. Compare BM25, vector, and hybrid — which still finds the right result?

**Exercise 3 — Your own domain:** Replace DOCS with 10 sentences from a domain you know. Pick 2 queries where BM25 should win (exact names) and 2 where vector should win (paraphrases). Verify hybrid covers both.

## What's next

- **26-rag-fusion** — RRF applied to parallel query variants (different angle on the same idea)
- **25-adaptive-rag** — route queries to the right retrieval strategy instead of always combining
- **24-graph-rag** — when even hybrid search misses multi-hop relational queries